# RNN 

1. Recurrent Neural Networks
    - 일반 신경망(MLP, CNN) : 입,출력의 벡터 크기가 항상 고정되어 있음. 또한, 계산 단계(e.g., 모델 내 레이어 수) 또한 고정되어 있음.  
        - computational steps의 제한? MLP는 입력 -> 은닉 -> 은닉 -> 출력. 이런식으로 연산 단계가 3단계에서 끝남.(데이터의 복잡성과 상관 없이 고정된 수의 레이어 사용) But RNN은 Loop를 돌기 때문에 데이터가 5개면 5번 돌고, 100개면 100번 수행.(데이터의 길이에 따라 모델이 스스로 연산 횟수 조절)
    - RNN : sequence of vector를 다룰 수 있음. (입,출력 상관 없이)  

        <img src="./RNN.jpeg" style="width: 80%;">   

        1) Vanilla mode (one to one) : RNN이 없는 처리 방식으로, 고정된 크기의 입력에서 고정된 크기의 출력을 내보냄.  
                                        (e.g. image classification)  
        2) Sequence output: 출력 벡터에 제한 X  (입력 1개, 출력 N개)
                            (e.g. image captioning takes an image and outputs a sentence of words)
        3) Sequence input: 입력 벡터에 제한 X  (입력 N개, 출력 1개)
                            (e.g. sentiment analysis where a given sentence is classified as expressing positive or negative sentiment)
        4) Sequence input and sequence output: computational steps에 제한 x  (비동기. 입력을 다 읽은 후 출력을 뱉음)
                                               (e.g. Machine Translation: an RNN reads a sentence in English and then outputs a sentence in French)
        5) Synced sequence input and output: 모든 사항에 대해 제한 X  (동기. 입력이 들어오는 족족 출력)
                                            (e.g. video classification where we wish to label each frame of the video)  

        - RNN은 State vector를 학습된 함수와 결합하여 새로운 state vector를 만들어냄 
            - 프로그래밍 관점에서, 이는 입력값과 내부 변수를 가지고 고정된 프로그램을 실행하는 것과 같음 (=> RNN 자체가 Algorithm적임)
            - 일반 신경망 훈련 = '함수'에 대한 최적화 / RNN 훈련 = '프로그램' 최적화

            

In [1]:
import torch 
import torch.nn as nn 
import torch.optim as optim 
import numpy as np 

In [2]:
# 1. 데이터 준비 (딕셔너리)
chars = ["h", "e", "l", "o"]
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(char_to_idx)
print(idx_to_char)

{'h': 0, 'e': 1, 'l': 2, 'o': 3}
{0: 'h', 1: 'e', 2: 'l', 3: 'o'}


In [3]:
vocab_size = len(chars)

In [4]:
# hello에서 hell(input) -> ello(output) 다음 글자 예측
input_str = "hell"
target_str = "ello"

x_data = [char_to_idx[c] for c in input_str]
y_data = [char_to_idx[c] for c in target_str]

print(x_data)
print(y_data)

[0, 1, 2, 2]
[1, 2, 2, 3]


In [5]:
# One-hot 인코딩 (입력벡터 x 생성)
x_np = np.array([np.eye(vocab_size)[x] for x in x_data])
x_one_hot = torch.FloatTensor(x_np)
y_data = torch.LongTensor(y_data)

print(x_one_hot)
print(y_data)

tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 1., 0.]])
tensor([1, 2, 2, 3])


$h_t = \tanh ( W_{hh}·h_{t-1} + W_{xh}·x_t )$

- $W_{xh}·x_t$: 입력 처리(현재 들어온 글자를 모델 은닉층 크기에 맞게 변환)
- $W_{hh}·h_{t-1}$: 기억 처리(전 단계에서 넘어온 기억을 현재 단계에 얼마나 반영할 지)

e.g.) x = (1,4)(hell중 h만 (한 글자씩 반복)), hidden size = 8 
1. 현재 입력 처리($W_{xh}·x_t$)  
$(1,4) \times  (4,8) = (1,8)$

2. 과거 기억 처리($W_{hh}·h_{t-1}$)  
**학습 시작 시 h = torch.zeros(1, 8)로 h(메모리)를 생성!!**   
$(1,8) \times  (8,8) = (1,8)$

3. 통합 및 압축
1.과 2.를 더한 뒤, tanh을 통과시켜 (1,8)의 새로운 $h_t$ 생성


In [12]:
# 2. RNN model 

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNN, self).__init__()
        self.hidden_size = hidden_size 
        
        # weight
        self.W_xh = nn.Linear(input_size, hidden_size)
        self.W_hh = nn.Linear(hidden_size, hidden_size)
        self.W_hy = nn.Linear(hidden_size, output_size)
        self.tanh = nn.Tanh()
        
    def forward(self, x, h):
        # h = tanh(W_hh * h + W_xh * x)
        h = self.tanh(self.W_xh(x) + self.W_hh(h))
        # y = W_hy * h
        y = self.W_hy(h)
        return y, h

In [17]:
model = RNN(input_size=4, hidden_size=8, output_size=4)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.1)

In [18]:
# train
for epoch in range(100):
    optimizer.zero_grad()
    h = torch.zeros(1, 8) # 매 문장 시작 시 hidden state 초기화 (0으로 시작)
    loss = 0
    
    # "h" -> "e" -> "l" -> "l" 순차적으로 입력 (loop)
    for i in range(len(x_one_hot)):
        input_x = x_one_hot[i].view(1,-1) # (4,1) -> (1,4)
        output, h = model(input_x, h)
        loss += criterion(output, y_data[i].view(1))
        
    loss.backward()
    optimizer.step() 
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}")

# test 
with torch.no_grad():
    h = torch.zeros(1, 8)
    result_str = ""
    for i in range(len(x_one_hot)):
        input_x = x_one_hot[i].view(1, -1)
        output, h = model(input_x, h)
        idx = output.argmax().item() 
        result_str += idx_to_char[idx]
        
print(f"입력: {input_str} -> 예측: {result_str}")

Epoch [20/100], Loss: 0.0118
Epoch [40/100], Loss: 0.0010
Epoch [60/100], Loss: 0.0006
Epoch [80/100], Loss: 0.0005
Epoch [100/100], Loss: 0.0004
입력: hell -> 예측: ello


# Paul Graham generator - Shakespeare
1. 전체적인 로직 (Step-by-Step)
- 데이터 로드: .txt 파일을 읽어 전체 문자를 파악
- 전처리: 모든 문자를 숫자로 바꾸고(인덱싱), 학습 데이터를 일정한 길이(예: 100글자)로 자름
- 모델 구축: RNN 레이어를 사용해 문맥을 더 길게 기억하도록 제작
- 학습: 다음 글자를 맞히도록 훈련
- 생성 (Sampling): 모델에게 첫 글자를 주고, 모델이 뱉은 글자를 다시 입력으로 넣어 셰익스피어 풍의 문장을 생성

In [25]:
import torch 
import torch.nn as nn 
import torch.optim as optim 
import numpy as np 
import torch.nn.functional as F

In [27]:
with open('./rnn_input.txt', 'r', encoding='utf-8') as f :
    text = f.read()
    
chars = sorted(list(set(text))) # 전체 고유 문자 
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

# text to number
data = torch.LongTensor([char_to_idx[c] for c in text])

print('chars:', chars)
print('vocab_size:', vocab_size)
print('char_to_idx: ', char_to_idx)
print('data: ', data[:5])

chars: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
vocab_size: 65
char_to_idx:  {'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
data:  tensor([18, 47, 56, 57, 58])


In [29]:
# hyperparam

rnn_size = 512 # hidden size
n_layers = 3  # 3층 RNN
seq_length = 100 # 한 번에 볼 문맥 길이
batch_size = 100
lr = 0.002
epochs = 50

In [32]:
class ShakespeareRNN(nn.Module):
    def __init__(self, vocab_size, rnn_size, n_layers, drop_out=0.5):
        super(ShakespeareRNN, self).__init__()
        self.vocab_size = vocab_size
        self.rnn_size = rnn_size
        self.n_layers = n_layers 
        
        # weights list (i2h, h2h를 저장할 리스트)
        self.i2h = nn.ModuleList() # input to hidden
        self.h2h = nn.ModuleList() # hidden to hidden 
        
        for L in range(n_layers):
            # input layer 의 input size = vocab_size(one-hot size), 이외의 층들은 rnn_size를 받음
            input_dim = vocab_size if L == 0 else rnn_size
            self.i2h.append(nn.Linear(input_dim, rnn_size))
            self.h2h.append(nn.Linear(rnn_size, rnn_size))
            
        self.dropout = nn.Dropout(drop_out)
        
        # 최종 출력층 : 은닉층의 결과를 다시 문자 크기로 (원핫인코딩을 인덱스로)
        self.decoder = nn.Linear(rnn_size, vocab_size)
            
    def forward(self, x, prev_h_states):
        # x: 현재 입력 글자의 인덱스
        # prev_h_states: 각 층의 이전 hidden state들이 담긴 리스트 [h1, h2, h3]
        
        # 원-핫 인코딩 (메모리 때문에 모델 내부에서 진행)
        x_one_hot = F.one_hot(x, num_classes=self.vocab_size).float()
        
        next_h_states = []
        current_input = x_one_hot 
        
        for L in range(self.n_layers):
            # RNN tich: h = tanh(i2h(x) + h2h(prev_h))
            i2h_part = self.i2h[L](current_input)
            h2h_part = self.h2h[L](prev_h_states[L])
            
            h_L = torch.tanh(i2h_part + h2h_part)
            
            # 마지막층 제외하고 나머지는 드롭아웃
            if L < self.n_layers - 1: 
                current_input = self.dropout(h_L)
                
            else:
                current_input = h_L
                
            next_h_states.append(h_L)
            
        output = self.decoder(self.dropout(current_input))
        return output, next_h_states
        

In [33]:
model = ShakespeareRNN(vocab_size, rnn_size, n_layers)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

In [34]:
print("학습 시작...")
for epoch in range(epochs):
    # 매 에포크마다 초기 hidden state 생성 (0벡터 리스트)
    h = [torch.zeros(1, rnn_size) for _ in range(n_layers)]
    
    total_loss = 0
    # 텍스트를 seq_length만큼씩 잘라서 학습
    for i in range(0, len(data) - seq_length, seq_length):
        inputs = data[i:i+seq_length]
        targets = data[i+1:i+seq_length+1] # 정답은 다음 글자
        
        optimizer.zero_grad()
        loss = 0
        
        # RNN은 한 글자씩 읽으며 h를 갱신.
        for t in range(seq_length):
            # h를 모델에 다시 집어넣어 순환.
            output, h = model(inputs[t].view(1), h)
            loss += criterion(output, targets[t].view(1))
            
            # 여기서 중요! h를 '복사'하는 게 아니라 '데이터만' 남겨서 
            # 다음 글자 연산 시 메모리 폭발을 방지합니다 (detach)
            h = [state.detach() for state in h]

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/(len(data)//seq_length):.4f}")

학습 시작...


KeyboardInterrupt: 

In [ ]:
def generate_text(start_str="ROMEO: ", length=200, temp=0.8):
    model.eval()
    with torch.no_grad():
        # 초기 설정
        chars_input = [char_to_idx[c] for c in start_str]
        h = [torch.zeros(1, rnn_size) for _ in range(n_layers)]
        
        # 시작 문자열로 우선 h를 예열(Warm-up)합니다.
        for i in range(len(chars_input) - 1):
            _, h = model(torch.tensor([chars_input[i]]), h)
        
        result = start_str
        current_char_idx = torch.tensor([chars_input[-1]])
        
        for _ in range(length):
            output, h = model(current_char_idx, h)
            
            # Softmax + Temperature 적용 (창의성 조절)
            output_dist = output.data.view(-1).div(temp).exp()
            top_idx = torch.multinomial(output_dist, 1)[0].item()
            
            result += idx_to_char[top_idx]
            current_char_idx = torch.tensor([top_idx])
            
    return result

print(generate_text())